# Setup

In [ ]:
from google.colab import drive
drive.mount('[REDACTED]')

In [ ]:
import os
import csv

In [ ]:
def collect_file_titles(folder_path):
    file_info = []  # Initialize an empty list to store file paths and titles

    # Traverse the folder and its subfolders
    for root, _, files in os.walk(folder_path):
        for file in files:
            # Get the full file path
            full_path = os.path.join(root, file)
            # Get the file title without the extension
            file_title = os.path.splitext(file)[0]
            # Store as a tuple (file title, full path)
            file_info.append((file_title, full_path))

    return file_info

def load_csv_as_tuples(file_path):
    data = []  # List to store tuples
    with open(file_path, "r", newline="", encoding="utf-8") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            data.append((row[0], row[1]))
    return data

In [ ]:
[REDACTED] = "[REDACTED]"
text_resources_folder = "[REDACTED]"
doc_and_word_map = "[REDACTED]"
original_resources = collect_file_titles([REDACTED])
text_resources = collect_file_titles(text_resources_folder)
map_data = load_csv_as_tuples(doc_and_word_map)

# Check keyword/phrase in the map_data
### If a unigram/bigram shows up in a presentation slide deck or webinar recording but cannot be found in the map_data, this indicates that the keyword/phrase got lost during the PDF to text or MP4 to text conversion process.

## Helper function

In [ ]:
def find_docs_in_map_data(map_data, keyword):
    # Find document names containing the keyword
    doc_names = [
        doc_name
        for doc_name, key in map_data
        if key == keyword
    ]

    # Remove duplicates
    map_docs = list(set(doc_names))

    # Print the document names
    if map_docs:
        print(f"Documents containing the keyword '{keyword}':")
        for name in map_docs:
            print(name)
    else:
        print(f"No matching documents found containing the keyword '{keyword}'.")

    print("Number of documents:", len(map_docs))

## Test your keyword/phrase here

In [ ]:
keyword = "governance council"
find_docs_in_map_data(map_data, keyword)

# Google Drive API

## Setup

In [ ]:
pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from googleapiclient.discovery import build
from google.oauth2 import service_account

## Helper functions

In [ ]:
def authenticate_drive_api():
    """Authenticate and return the Drive API service."""
    credentials = service_account.Credentials.[REDACTED](
        "[REDACTED]",
        scopes=["[REDACTED]"]
    )
    return build("drive", "v3", credentials=credentials)

In [ ]:
def list_files_in_folder(drive_service, folder_id):
    """List all files and subfolders in a specific folder by folder ID."""
    query = f"'{folder_id}' in parents"
    files = []
    page_token = None

    while True:
        results = drive_service.files().list(
            q=query,
            spaces="drive",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token
        ).execute()

        # Extend the files list with the current batch of results
        files.extend(results.get("files", []))

        # Update the page token to fetch the next page
        page_token = results.get("nextPageToken", None)
        if not page_token:
            break

    return files

def list_files_recursively(drive_service, folder_id):
    """Recursively list all files in a folder and its subfolders."""
    all_files = []
    items = list_files_in_folder(drive_service, folder_id)

    for item in items:
        if item["mimeType"] == "application/vnd.google-apps.folder":
            # If the item is a folder, recursively list its contents
            all_files.extend(list_files_recursively(drive_service, item["id"]))
        else:
            # If the item is a file, add it to the list
            all_files.append(item)

    return all_files

In [ ]:
# Authenticate
drive_service = authenticate_drive_api()

# Specify the folder ID directly
folder_id = "[REDACTED]"

try:
    # Collect all files in the specified folder and its subfolders
    all_files = list_files_recursively(drive_service, folder_id)

    # Print file details
    if all_files:
        print(f"All Files in folder with ID '{folder_id}' and its subfolders:")
        for file in all_files:
            print(f"Name: {file['name']}, ID: {file['id']}")
    else:
        print(f"No files found in the folder with ID '{folder_id}' or its subfolders.")
except Exception as e:
    print(f"An error occurred: {e}")
print("Number of documents collected:", len(all_files))

In [ ]:
def search_file_by_name(drive_service, file_name):
    """Search for a file by its name."""
    query = f"name = '{file_name}'"
    results = drive_service.files().list(
        q=query, spaces="drive", fields="files(id, name, parents, mimeType)"
    ).execute()
    return results.get("files", [])

# Search for the specific file
file_name = "The [REDACTED] Presents Strategic Data Governance – Leveraging a Federated Data Governance Model-20150107 1900-1-m9trhwstqd.mp4"
found_files = search_file_by_name(drive_service, file_name)

if found_files:
    for file in found_files:
        print(f"Name: {file['name']},       ID: {file['id']}, Parents: {file.get('parents')}")
else:
    print("File not found.")

In [ ]:
def list_all_folders(drive_service):
    """List all folders in Google Drive."""
    query = "mimeType = 'application/vnd.google-apps.folder'"
    results = drive_service.files().list(
        q=query, spaces="drive", fields="files(id, name)"
    ).execute()
    return results.get("files", [])

# List all folders
folders = list_all_folders(drive_service)
for folder in folders:
    print(f"Folder Name: {folder['name']},      ID: {folder['id']}")

In [ ]:
def [REDACTED](files):
    """Generate Google Drive links for a list of files."""
    base_url = "[REDACTED]"
    links = []
    for file in files:
        file_link = f"{base_url}{file['id']}/view"
        links.append({"name": file["name"], "link": file_link})
    return links

# Generate links for the retrieved files
drive_links = [REDACTED](all_files)

# Print the links
if drive_links:
    print("Google Drive Links:")
    for item in drive_links:
        print(f"Name: {item['name']}, Link: {item['link']}")
else:
    print("No files to generate links for.")
print("Number of links generated:", len(drive_links))

In [ ]:
def [REDACTED](drive_links, map_data):
    """
    Match files in drive_links with those in map_data, considering cases where
    map_data file names may end with "_word_count" or "_bigram_count".
    """
    # Normalize document names in map_data
    normalized_map_data = [
        (os.path.splitext(doc_name.replace("_word_count", "").replace("_bigram_count", ""))[0], keyword)
        for doc_name, keyword in map_data
    ]

    # Match drive_links with map_data
    matched_items = []
    for drive_link in drive_links:
        drive_doc_name = os.path.splitext(drive_link["name"])[0]  # Normalize drive link name
        for map_doc_name, keyword in normalized_map_data:
            if drive_doc_name == map_doc_name or f"{drive_doc_name}_word_count" == map_doc_name or f"{drive_doc_name}_bigram_count" == map_doc_name:
                matched_items.append({
                    "name": drive_link["name"],
                    "link": drive_link["link"],
                    "keyword": keyword
                })

    return matched_items


In [ ]:
def [REDACTED](keyword, drive_links, map_data):

    # Filter map_data to include only entries for the given keyword
    filtered_map_data = list({
        (doc_name, key) for doc_name, key in map_data if key == keyword
    })

    # Match drive links with the filtered map_data
    return [REDACTED](drive_links, filtered_map_data)

In [ ]:
def fetch_links(keyword, drive_links, map_data):
    # Fetch documents and links for the keyword
    documents_with_links = [REDACTED](keyword, drive_links, map_data)

    # Print results
    if documents_with_links:
        num_of_docs = len(documents_with_links)
        for doc in documents_with_links:
            print(f"Name: {doc['name']},      Link: {doc['link']}")
        print(f"Documents containing the keyword '{keyword}' are shown above.")
        print(f"Number of documents: {num_of_docs}.")
    else:
        print(f"No documents found for the keyword '{keyword}'.")

## Fetching document names and links for a specified unigram/bigram

In [ ]:
keyword = "governance council"
fetch_links(keyword, drive_links, map_data)